## Ollama Agent Chat test notebook

In [16]:
from dotenv import load_dotenv
load_dotenv()

True

In [17]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent
import os

llm = ChatOllama(model="gpt-oss:20b", base_url=os.getenv("OLLAMA_BASE_URL"), temperature=0.0)

system_prompt = """You are a helpful assistant. Whenever the user asks you questions about current events, use your web search tools to obtain the most up to date information."""

agent = create_agent(llm, system_prompt=system_prompt)


### Without a websearch tool to obtain current information, the agent's answers may be out of date

In [18]:
from langchain_core.messages import HumanMessage

response = agent.invoke({
    "messages": [HumanMessage(content="Who is the current mayor of New York City?")]
})

print(f"Current mayor of NYC: {response["messages"][-1].content}")

Current mayor of NYC: 


In [19]:
response = agent.invoke({
    "messages": [HumanMessage(content="How up-to-date is your knowledge base?")]
})

print(response["messages"][-1].content)

My training data goes up to **June 2024**. That means I can provide information, context, and background that was publicly available up to that point. For anything that has happened or changed after that date—such as recent news, new regulations, product releases, or other time‑sensitive events—I can use the web‑search tool to fetch the latest details and incorporate them into my response.


### Defining simple tools

In [20]:
from langchain.tools import tool

@tool
def to_the_third_power(x: float) -> float:
    """Returns the provided number to the third power."""
    return x ** 3.0

In [21]:
agent = create_agent(
    model=llm,
    system_prompt=(
        "You are a master of arithmetic. "
        "Use your tools whenever the user asks to raise a number to the third power."
    ),
    tools=[to_the_third_power]
)

response = agent.invoke({
    "messages": [HumanMessage(content="What is 8.5 to the third power?")]
})

print(response["messages"][-1].content)

8.5 to the third power is **614.125**.


### Using provided tools

In [22]:
from langchain_tavily import TavilySearch

tavily_search = TavilySearch()

agent = create_agent(model=llm, system_prompt=system_prompt, tools=[tavily_search])

In [23]:
response = agent.invoke({
    "messages": [HumanMessage(content="Who is the current mayor of New York City?")]
})

print(response["messages"][-1].content)

The current mayor of New York City is **Zohran Mamdani**. He was sworn in as the 112th mayor on January 1, 2026 and has been serving in that role since then【3†L1-L4】【3†L5-L8】.


In [24]:
from pprint import pprint

for msg in response["messages"]:
    pprint(msg)
    print("==================================")

HumanMessage(content='Who is the current mayor of New York City?', additional_kwargs={}, response_metadata={}, id='eb2489f8-d1bb-4637-b460-d075b841979c')
AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gpt-oss:20b', 'created_at': '2026-03-29T16:03:56.7477965Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1396713000, 'load_duration': 196754700, 'prompt_eval_count': 1289, 'prompt_eval_duration': 466608700, 'eval_count': 81, 'eval_duration': 688681500, 'logprobs': None, 'model_name': 'gpt-oss:20b', 'model_provider': 'ollama'}, id='lc_run--019d3a56-6b55-7551-a7b3-771acc946d34-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'current mayor of New York City', 'search_depth': 'basic', 'include_images': False}, 'id': '9801a55b-fa30-4c8b-be52-23c7cc17fa50', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1289, 'output_tokens': 81, 'total_tokens': 1370})
ToolMessage(content='{"query": "current mayor of New York City", "